# Baseline Model



In [9]:
# login Hugging Face Hub
import os
from dotenv import load_dotenv

load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN) 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [10]:
from datasets import load_dataset
import os

# Load the real dataset from HuggingFace
ds = load_dataset("SleepyJesse/ai_music_tiny", split="train", cache_dir="./data")
print(f"Real dataset loaded with {len(ds)} samples")

Real dataset loaded with 1000 samples


Extract mel frequency cepstral coefficients

In [11]:
import librosa
import numpy as np

def extract_mfcc_features(audio_path, sr=22050, duration=5, n_mfcc=20):
    # Load audio file using librosa
    y, sr = librosa.load(audio_path, sr=sr, duration=duration)
    
    if len(y) < sr * duration:  # pad with zeros if shorter than desired duration
        y = np.pad(y, (0, sr * duration - len(y)))
    else:  # truncate if longer
        y = y[:sr * duration]

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    mfcc_mean = mfcc.mean(axis=1)  # compute mean across time frames
    mfcc_std = mfcc.std(axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])

Build dataset

In [ ]:
def build_dataset(ds, max_samples_per_class=500):
    """Extract audio from HuggingFace dataset"""
    X, y = [], []
    class_counts = {0: 0, 1: 0}
    
    print("Building dataset from HuggingFace...")
    
    for idx in range(len(ds)):
        try:
            sample = ds[idx]
            label_str = sample.get('label', 'unknown')
            label = 0 if label_str == 'human' else 1
            
            if class_counts[label] >= max_samples_per_class:
                continue
            
            try:
                audio_obj = sample.get('audio')
                if audio_obj and hasattr(audio_obj, 'array'):
                    audio_array = audio_obj.array
                    sr = audio_obj.sampling_rate
                    
                    # Process the audio array
                    y_audio = audio_array
                    if len(y_audio) < sr * 5:
                        y_audio = np.pad(y_audio, (0, sr * 5 - len(y_audio)))
                    else:
                        y_audio = y_audio[:sr * 5]
                    
                    # Extract MFCC
                    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
                    mfcc_mean = mfcc.mean(axis=1)
                    mfcc_std = mfcc.std(axis=1)
                    features = np.concatenate([mfcc_mean, mfcc_std])
                    
                    X.append(features)
                    y.append(label)
                    class_counts[label] += 1
                    
                    if (class_counts[0] + class_counts[1]) % 100 == 0:
                        print(f"  Processed {class_counts[0] + class_counts[1]} samples...")
            except Exception as e:
                if idx < 5:
                    print(f"Error processing audio in sample {idx}: {str(e)[:60]}")
                continue
                
        except Exception as e:
            if idx < 5:
                print(f"Error accessing sample {idx}: {str(e)[:60]}")
            continue
        
        if all(c >= max_samples_per_class for c in class_counts.values()):
            break
    
    if len(X) == 0:
        raise ValueError("Could not process any samples from HuggingFace dataset. Please check the dataset and audio decoding.")
    
    print(f"Collected {len(X)} real samples (human: {class_counts[0]}, ai: {class_counts[1]})")
    return np.array(X), np.array(y)

X, y = build_dataset(ds)

Building dataset from HuggingFace...
Error accessing sample 0: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 1: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 2: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 3: Could not load libtorchcodec. Likely causes:
          1. FF
Error accessing sample 4: Could not load libtorchcodec. Likely causes:
          1. FF


ValueError: Could not process any samples from HuggingFace dataset. Please check the dataset and audio decoding.

Train model

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# X, y already built from previous cell

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.42
              precision    recall  f1-score   support

           0       0.41      0.39      0.40       100
           1       0.42      0.45      0.44       100

    accuracy                           0.42       200
   macro avg       0.42      0.42      0.42       200
weighted avg       0.42      0.42      0.42       200

